<a href="https://colab.research.google.com/github/bmfmancini/edureka_pgp_ai_ml/blob/main/AgenticAI_5thApril.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### we are working on recruitment demo problem
# step 1 - lets all neccesar packages
!pip install -q langchain langchain-google-genai langchain-community python-dotenv pypdf docx2txt langchain_core langchain-text-splitters langchain-classic
### step 2 - import libraries
import os
from google.colab import files, userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_core.output_parsers import StrOutputParser
# step 3- class API key
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
### step 4 LLM and output parser
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)
output_parser = StrOutputParser()
### step 5 - Define prompt
template = """You are an AI Resume Screener
Compare the following resume with the job description
Job Description:{job_desc}
Resume Text:{resume_text}
Give a simple analysis with:
- Fit Score ( 0-100 )
- Domian Relevant Knowledge
- Top 5 matching skills
- Missing important skills
- One-line Verdict
"""
prompt = PromptTemplate(template=template, input_variables=["job_desc", "resume_text"])
chain = LLMChain(llm=llm, prompt=prompt, output_parser=output_parser)
### step 6 - Upload and Extract Resume text
print("Plese upload your resume file (PDF/DOCX/TXT):")
uploaded = files.upload()
if not uploaded:
  raise ValueError("No file uploaded.")
resume_path = list(uploaded.keys())[0]
# Detect file type and load
if resume_path.lower().endswith('.pdf'):
  loader = PyPDFLoader(resume_path)
elif resume_path.lower().endswith('.docx') or resume_path.lower().endswith('.doc'):
  loader = Docx2txtLoader(resume_path)
else:
  loader = TextLoader(resume_path, encoding="utf-8" )
docs = loader.load()
# split large text into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
resume_text = " ".join(c.page_content for c in chunks).strip()
# safety fallback condition
if not resume_text:
  raise ValueError("No resume text found.")
### step 7 - run the agent and give job description
job_description = """ We are hiring a senior AI engineer who is skillled in Python, SQL, Cloud(AWS/GCP/Azure), Data Analysis and Langchain."""
result = chain.run(job_desc=job_description, resume_text=resume_text)
print(result)